# Experiments on ETTh1 and ETTh2

Этот ноутбук запускает три подхода (LSTM + Optuna, Informer, LLM-based architecture search) на датасетах ETTh1 и ETTh2 и строит сравнение по метрике MSE.

In [1]:
from __future__ import annotations

import ast
import logging
import os
import sys
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

project_root = Path.cwd().resolve()
src_dir = project_root

if not src_dir.exists():
    raise RuntimeError(f'Expected src directory at {src_dir}, but it does not exist.')

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from experiments import (
    run_informer_etth_experiment,
    ExperimentResult,
)
from experiments.datasets import load_ett_csv_dataset
from edlm_search.baseline_optuna import run_optuna_for_etth
from edlm_search.ett_evaluator import ETTM1Evaluator
from edlm_search.llm_clients import DeepSeekClient, LMStudioClient
from edlm_search.problem import Problem
from edlm_search.runner import UnsafeRunner
from edlm_search.sampler import CandidateSampler
from edlm_search.search_loop import LLMBasedArchitectureSearch

logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(name)s - %(message)s',
)
logger = logging.getLogger('etth_experiments')

logger.info(f'Project root: {project_root}')
logger.info(f'src dir: {src_dir}')

/Users/roman/Projects/PycharmProjects/ITMO/edlm_search/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-11-19 00:56:21,523 - INFO - etth_experiments - Project root: /Users/roman/Projects/PycharmProjects/ITMO/edlm_search/src
2025-11-19 00:56:21,524 - INFO - etth_experiments - src dir: /Users/roman/Projects/PycharmProjects/ITMO/edlm_search/src


In [2]:
@dataclass
class DatasetPaths:
    """Absolute paths to CSV files of the ETTh1 and ETTh2 datasets."""

    etth1_csv_path: Path
    etth2_csv_path: Path


@dataclass
class SplitConfig:
    """Train/validation split configuration for ETTh datasets."""

    max_rows: int
    train_ratio: float


@dataclass
class OptunaConfig:
    """Hyperparameters for the LSTM Optuna baseline search."""

    seq_len: int
    pred_len: int
    num_epochs: int
    n_trials: int


@dataclass
class LLMSearchConfig:
    """Configuration of the LLM-based architecture search loop."""

    num_epochs: int
    max_candidates: int
    top_k_for_crossover: int
    metric_name: str


@dataclass
class InformerConfig:
    """Configuration for running Informer via the external wrapper script."""

    num_epochs: int


@dataclass
class LLMProviderConfig:
    """Configuration of the LLM provider (LM Studio or DeepSeek)."""

    provider: str
    base_url: str
    model_name: str
    api_key_env_var: str


dataset_paths = DatasetPaths(
        etth1_csv_path=(project_root / 'ETDataset' / 'ETT-small' / 'ETTh1.csv'),
        etth2_csv_path=(project_root / 'ETDataset' / 'ETT-small' / 'ETTh2.csv'),
)

split_config = SplitConfig(
        max_rows=int(os.environ.get('MAX_ROWS', '10000')),
        train_ratio=float(os.environ.get('TRAIN_RATIO', '0.8')),
)

optuna_config = OptunaConfig(
        seq_len=int(os.environ.get('SEQ_LEN', '96')),
        pred_len=int(os.environ.get('PRED_LEN', '24')),
        num_epochs=int(os.environ.get('NUM_EPOCHS', '5')),
        n_trials=int(os.environ.get('N_TRIALS', '20')),
)

llm_search_config = LLMSearchConfig(
        num_epochs=int(os.environ.get('LLM_NUM_EPOCHS', '5')),
        max_candidates=int(os.environ.get('MAX_CANDIDATES', '5')),
        top_k_for_crossover=int(os.environ.get('TOP_K_FOR_CROSSOVER', '3')),
        metric_name=os.environ.get('METRIC_NAME', 'mse'),
)

informer_config = InformerConfig(
        num_epochs=int(os.environ.get('INFORMER_NUM_EPOCHS', '10')),
)

provider_config = LLMProviderConfig(
        provider=os.environ.get('LLM_PROVIDER', 'lmstudio'),
        base_url=os.environ.get('LLM_BASE_URL', 'http://localhost:1234/v1'),
        model_name=os.environ.get('LLM_MODEL_NAME', 'your-lmstudio-model-name'),
        api_key_env_var='DEEPSEEK_API_KEY',
)

logger.info(f'LLM provider: {provider_config.provider}')
logger.info(f'ETTh1 path: {dataset_paths.etth1_csv_path}')
logger.info(f'ETTh2 path: {dataset_paths.etth2_csv_path}')

2025-11-19 00:56:21,532 - INFO - etth_experiments - LLM provider: lmstudio
2025-11-19 00:56:21,533 - INFO - etth_experiments - ETTh1 path: /Users/roman/Projects/PycharmProjects/ITMO/edlm_search/src/ETDataset/ETT-small/ETTh1.csv
2025-11-19 00:56:21,533 - INFO - etth_experiments - ETTh2 path: /Users/roman/Projects/PycharmProjects/ITMO/edlm_search/src/ETDataset/ETT-small/ETTh2.csv


In [3]:
def load_datasets(dataset_paths: DatasetPaths, split_config: SplitConfig):
    """Load and split ETTh1 and ETTh2 datasets into train and validation parts."""
    train_dfs: dict[str, pd.DataFrame] = {}
    valid_dfs: dict[str, pd.DataFrame] = {}

    for name, csv_path in (
            ('ETTh1', dataset_paths.etth1_csv_path),
            ('ETTh2', dataset_paths.etth2_csv_path),
    ):
        if not csv_path.exists():
            raise FileNotFoundError(f'Dataset file {name} not found at {csv_path}')
        train_df, valid_df = load_ett_csv_dataset(
                csv_path=str(csv_path),
                max_rows=split_config.max_rows,
                train_ratio=split_config.train_ratio,
        )
        logger.info(f'Dataset {name} loaded: train={len(train_df)}, valid={len(valid_df)}')
        train_dfs[name] = train_df
        valid_dfs[name] = valid_df

    return train_dfs, valid_dfs


train_dfs, valid_dfs = load_datasets(dataset_paths=dataset_paths, split_config=split_config)

2025-11-19 00:56:21,555 - INFO - etth_experiments - Dataset ETTh1 loaded: train=8000, valid=2000
2025-11-19 00:56:21,570 - INFO - etth_experiments - Dataset ETTh2 loaded: train=8000, valid=2000


In [4]:
optuna_results: dict[str, dict[str, object]] = {}

for dataset_name in ('ETTh1', 'ETTh2'):
    train_df = train_dfs[dataset_name]
    valid_df = valid_dfs[dataset_name]

    logger.info(
            f'[{dataset_name}] Starting Optuna baseline: seq_len={optuna_config.seq_len}, '
            f'pred_len={optuna_config.pred_len}, epochs={optuna_config.num_epochs}, '
            f'trials={optuna_config.n_trials}'
    )

    study = run_optuna_for_etth(
            train_df=train_df,
            valid_df=valid_df,
            seq_len=optuna_config.seq_len,
            pred_len=optuna_config.pred_len,
            num_epochs=optuna_config.num_epochs,
            n_trials=optuna_config.n_trials,
            target_column='OT',
    )

    best_mse = float(study.best_value)
    optuna_results[dataset_name] = {
        'study': study,
        'best_mse': best_mse,
        'best_params': dict(study.best_trial.params),
    }

    logger.info(
            f'[{dataset_name}] Optuna baseline finished. Best MSE: {best_mse}, '
            f'best params: {study.best_trial.params}'
    )

2025-11-19 00:56:21,577 - INFO - etth_experiments - [ETTh1] Starting Optuna baseline: seq_len=96, pred_len=24, epochs=5, trials=20
[I 2025-11-19 00:56:21,578] A new study created in memory with name: etth_lstm_mse
2025-11-19 00:56:21,999 - INFO - zeus.device.gpu.nvidia - pynvml is available but could not initialize NVML: NVML Shared Library Not Found.
2025-11-19 00:56:22,004 - WARNING - zeus.device.gpu.amd - Failed to import amdsmi due to a key error on: ['libamd_smi.so']. Ensure that amdsmi is installed on your system.
2025-11-19 00:56:22,005 - INFO - zeus.device.cpu.rapl - RAPL is not supported on this CPU.
2025-11-19 00:56:22,005 - INFO - zeus.monitor.energy - Monitoring GPU indices [0].
2025-11-19 00:56:22,005 - INFO - zeus.monitor.energy - Monitoring CPU indices []
[W 2025-11-19 00:56:22,005] Trial 0 failed with parameters: {'hidden_size': 106, 'num_layers': 1, 'learning_rate': 0.00039478776457894455, 'batch_size': 64} because of the following error: ValueError('No GPUs available.

ValueError: No GPUs available.

In [5]:
informer_results: dict[str, ExperimentResult] = {}

for dataset_name, csv_path in (
        ('ETTh1', dataset_paths.etth1_csv_path),
        ('ETTh2', dataset_paths.etth2_csv_path),
):
    if not csv_path.exists():
        logger.info(f'[{dataset_name}] Skipping Informer run: file {csv_path} not found.')
        continue

    metrics_json_path = project_root / 'artifacts' / f'informer_{dataset_name.lower()}_metrics.json'
    informer_script = src_dir / 'Informer2020' / 'informer_experiment_wrapper.py'

    logger.info(
            f'[{dataset_name}] Running Informer: script={informer_script}, csv={csv_path}, '
            f'epochs={informer_config.num_epochs}'
    )

    result = run_informer_etth_experiment(
            dataset_name=dataset_name,
            csv_path=str(csv_path),
            informer_script_path=str(informer_script),
            metrics_json_path=str(metrics_json_path),
            extra_args=['--epochs', str(informer_config.num_epochs), '--data', dataset_name],
            model_name=f'informer-{dataset_name.lower()}',
    )
    informer_results[dataset_name] = result
    logger.info(f'[{dataset_name}] Informer finished. Metrics: {result.metrics}')

2025-11-19 00:56:42,567 - INFO - etth_experiments - [ETTh1] Running Informer: script=/Users/roman/Projects/PycharmProjects/ITMO/edlm_search/src/Informer2020/informer_experiment_wrapper.py, csv=/Users/roman/Projects/PycharmProjects/ITMO/edlm_search/src/ETDataset/ETT-small/ETTh1.csv, epochs=10
2025-11-19 00:56:42,568 - INFO - experiments.experiment_api - Запуск Informer-эксперимента для датасета "ETTh1" с CSV по пути "/Users/roman/Projects/PycharmProjects/ITMO/edlm_search/src/ETDataset/ETT-small/ETTh1.csv".


RuntimeError: Скрипт Informer завершился с кодом 1. stdout="" stderr="Traceback (most recent call last):
  File "/Users/roman/Projects/PycharmProjects/ITMO/edlm_search/src/Informer2020/informer_experiment_wrapper.py", line 320, in <module>
    main()
  File "/Users/roman/Projects/PycharmProjects/ITMO/edlm_search/src/Informer2020/informer_experiment_wrapper.py", line 311, in main
    metrics = _run_informer_and_get_metrics(
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/roman/Projects/PycharmProjects/ITMO/edlm_search/src/Informer2020/informer_experiment_wrapper.py", line 246, in _run_informer_and_get_metrics
    gpu_monitor = ZeusMonitor(gpu_indices=[0])
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/roman/Projects/PycharmProjects/ITMO/edlm_search/.venv/lib/python3.12/site-packages/zeus/monitor/energy.py", line 251, in __init__
    if not self.gpus.supports_get_total_energy_consumption(gpu_index)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [ ]:
def create_llm_pipeline(provider_config: LLMProviderConfig):
    """Create an LLMPipeline instance for the configured provider."""
    provider_lower = provider_config.provider.lower()

    if provider_lower == 'lmstudio':
        client = LMStudioClient(
                base_url=provider_config.base_url,
                model_name=provider_config.model_name,
        )
    elif provider_lower == 'deepseek':
        api_key = os.environ.get(provider_config.api_key_env_var)
        if not api_key:
            raise RuntimeError(
                    f"Provider 'deepseek' requires environment variable {provider_config.api_key_env_var} to be set."
            )
        client = DeepSeekClient(
                api_key=api_key,
                base_url=provider_config.base_url,
                model_name=provider_config.model_name,
        )
    else:
        raise ValueError(f'Unknown LLM provider: {provider_config.provider}')

    return client.create_pipeline()


problem_dir = Path(os.environ.get('PROBLEM_DIR', './examples/et')).resolve()
if not problem_dir.exists():
    raise FileNotFoundError(f'Problem directory not found: {problem_dir}')

problem = Problem.from_directory(str(problem_dir))
logger.info('Problem statement successfully loaded.')

llm_pipeline = create_llm_pipeline(provider_config=provider_config)
sampler = CandidateSampler(llm_pipeline=llm_pipeline, problem=problem)

In [ ]:
async def run_llm_search_for_dataset(
        dataset_name: str,
        train_df: pd.DataFrame,
        valid_df: pd.DataFrame,
        llm_search_config: LLMSearchConfig,
        sampler: CandidateSampler,
):
    """Run LLM-based architecture search for a single dataset."""
    evaluator = ETTM1Evaluator(
            train_df=train_df,
            valid_df=valid_df,
            target_column='OT',
            metric_name=llm_search_config.metric_name,
            num_epochs=llm_search_config.num_epochs,
    )

    def runner_factory():
        return UnsafeRunner()

    search = LLMBasedArchitectureSearch(
            evaluator=evaluator,
            sampler=sampler,
            runner_factory=runner_factory,
            backend_name=provider_config.provider,
            metric_name=llm_search_config.metric_name,
            max_candidates=llm_search_config.max_candidates,
            top_k_for_crossover=llm_search_config.top_k_for_crossover,
    )

    logger.info(
            f'[{dataset_name}] Starting LLM-based architecture search: '
            f'max_candidates={llm_search_config.max_candidates}, '
            f'epochs_per_candidate={llm_search_config.num_epochs}'
    )

    database = await search.run_search()

    logger.info(
            f'[{dataset_name}] LLM-based search finished. Total candidates in database: {len(database.dataframe)}'
    )

    best_records = database.top_k_by_metric(
            metric_name=llm_search_config.metric_name,
            k=1,
    )
    best_record = best_records[0]
    best_mse = float(best_record.metrics[llm_search_config.metric_name])

    logger.info(
            f'[{dataset_name}] Best LLM candidate: id={best_record.candidate_id}, '
            f'{llm_search_config.metric_name}={best_mse}'
    )

    return database, best_mse

In [ ]:
llm_search_databases: dict[str, object] = {}
llm_best_mse: dict[str, float] = {}

for dataset_name in ('ETTh1', 'ETTh2'):
    train_df = train_dfs[dataset_name]
    valid_df = valid_dfs[dataset_name]

    database, best_mse = await run_llm_search_for_dataset(
            dataset_name=dataset_name,
            train_df=train_df,
            valid_df=valid_df,
            llm_search_config=llm_search_config,
            sampler=sampler,
    )
    llm_search_databases[dataset_name] = database
    llm_best_mse[dataset_name] = best_mse

logger.info('LLM-based architecture search completed for ETTh1 and ETTh2.')

artifacts_dir = project_root / 'artifacts'
artifacts_dir.mkdir(parents=True, exist_ok=True)

for dataset_name, database in llm_search_databases.items():
    csv_path = artifacts_dir / f'llm_search_{dataset_name.lower()}_candidates.csv'
    database.dataframe.to_csv(csv_path, index=False)
    logger.info(f'[{dataset_name}] LLM search database saved to {csv_path}')

In [ ]:
rows: list[dict[str, object]] = []

for dataset_name in ('ETTh1', 'ETTh2'):
    optuna_best_mse = optuna_results[dataset_name]['best_mse']
    rows.append(
            {
                'approach': 'lstm_optuna_best',
                'dataset': dataset_name,
                'mse': float(optuna_best_mse),
            }
    )

    informer_result = informer_results.get(dataset_name)
    informer_mse = None
    if informer_result is not None:
        informer_mse_raw = informer_result.metrics.get('mse')
        if informer_mse_raw is not None:
            informer_mse = float(informer_mse_raw)
    rows.append(
            {
                'approach': 'informer_original',
                'dataset': dataset_name,
                'mse': informer_mse if informer_mse is not None else np.nan,
            }
    )

    llm_mse = llm_best_mse[dataset_name]
    rows.append(
            {
                'approach': 'llm_search_best_candidate',
                'dataset': dataset_name,
                'mse': float(llm_mse),
            }
    )

comparison_df = pd.DataFrame(rows)
comparison_df.sort_values(by=['dataset', 'approach'], inplace=True)
comparison_df.reset_index(drop=True, inplace=True)

artifacts_dir = project_root / 'artifacts'
artifacts_dir.mkdir(parents=True, exist_ok=True)
comparison_csv_path = artifacts_dir / 'etth_comparison_metrics.csv'
comparison_df.to_csv(comparison_csv_path, index=False)
logger.info(f'Final comparison table saved to {comparison_csv_path}')

logger.info('Final comparison table between approaches for ETTh1 and ETTh2 is ready.')
comparison_df

In [ ]:
comparison_csv_path = project_root / 'artifacts' / 'etth_comparison_metrics.csv'
if not comparison_csv_path.exists():
    logger.info(f'Comparison CSV not found at {comparison_csv_path}, skipping bar plots.')
else:
    comparison_df = pd.read_csv(comparison_csv_path)
    for dataset_name in ('ETTh1', 'ETTh2'):
        subset = comparison_df[comparison_df['dataset'] == dataset_name]
        if subset.empty:
            logger.info(f'[{dataset_name}] Comparison subset is empty, skipping bar plot.')
            continue

        plt.figure(figsize=(6, 4))
        x_positions = np.arange(len(subset))
        plt.bar(x_positions, subset['mse'])
        plt.xticks(
                x_positions,
                subset['approach'],
                rotation=30,
                ha='right',
        )
        plt.ylabel('MSE')
        plt.title(f'Model comparison on {dataset_name}: MSE')
        plt.tight_layout()
        plt.show()

In [ ]:
for dataset_name in ('ETTh1', 'ETTh2'):
    csv_path = project_root / 'artifacts' / f'llm_search_{dataset_name.lower()}_candidates.csv'
    if not csv_path.exists():
        logger.info(f'[{dataset_name}] LLM search CSV not found at {csv_path}, skipping dynamics plot.')
        continue

    df = pd.read_csv(csv_path)
    if 'mse' not in df.columns:
        if 'metrics' not in df.columns:
            logger.info(
                    f'[{dataset_name}] Columns "mse" and "metrics" are missing, skipping plot.'
            )
            continue
        try:
            df['mse'] = df['metrics'].apply(
                    lambda x: float(ast.literal_eval(x).get('mse')) if isinstance(x, str) else float('nan')
            )
        except Exception as exc:
            logger.info(f'[{dataset_name}] Failed to parse "metrics" column for MSE: {exc}')
            continue

    plt.figure(figsize=(6, 4))
    plt.plot(range(1, len(df) + 1), df['mse'])
    plt.xlabel('Candidate index')
    plt.ylabel('MSE')
    plt.title(f'LLM search dynamics on {dataset_name}: MSE per candidate')
    plt.grid(True)
    plt.tight_layout()
    plt.show()